# 08. 特徴ベクトルの最終組み立て

`01`〜`07`で確定した処理（RIASEC補完、認知度ラベル、PCA、質問の採点ロジック）を
一つの成果物にまとめる。ここで作るのは2つ。

1. `data/processed/jobs.csv` — 推薦アルゴリズムが実際に使う167職業のテーブル
   （RIASEC・PC1〜4・認知度・解説文）
2. `data/processed/riasec_transform.json` — 質問の採点に必要な変換パラメータ
   （標準化・PCA・質問定義）。APIはnotebookに依存せず、この2ファイルだけで
   採点〜推薦ができるようにする


In [1]:
import json
import re

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from ipd_loader import load_description, load_numeric

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

desc, desc_labels = load_description()
num, num_labels = load_numeric()
names = num[num.columns[1]]


## RIASEC補完（02と同じ手順）

In [2]:
riasec_cols = [c for c in num.columns if re.match(r"IPD_04_01_", str(c))]
know_cols = [c for c in num.columns if re.match(r"IPD_04_04_01_", str(c))]
work_cols = [c for c in num.columns if re.match(r"IPD_04_05_", str(c))]

riasec = num[riasec_cols].apply(pd.to_numeric, errors="coerce")
know = num[know_cols].apply(pd.to_numeric, errors="coerce")
work = num[work_cols].apply(pd.to_numeric, errors="coerce")

riasec_observed = ~riasec.isna().all(axis=1)
no_input = know.isna().all(axis=1) & work.isna().all(axis=1)
predictable = (~riasec_observed) & ~no_input

def scale_domain(domain_df):
    scaler = StandardScaler()
    filled = domain_df.fillna(domain_df.mean())
    scaled = pd.DataFrame(scaler.fit_transform(filled), columns=domain_df.columns, index=domain_df.index)
    return scaled.fillna(0.0)

know_scaled = scale_domain(know)
work_scaled = scale_domain(work)
X_all = pd.concat([know_scaled, work_scaled], axis=1)
X_train = X_all[riasec_observed]
y_train = riasec[riasec_observed]
riasec_model = Ridge(alpha=1.0).fit(X_train.values, y_train.values)

riasec_full = riasec.copy()
X_pred = X_all[predictable]
riasec_full.loc[predictable, riasec_cols] = riasec_model.predict(X_pred.values)

unavailable = (~riasec_observed) & no_input
riasec_source = pd.Series("observed", index=num.index)
riasec_source[predictable] = "predicted"
riasec_source[unavailable] = "unavailable"
riasec_source.value_counts()


observed       482
predicted       29
unavailable      7
Name: count, dtype: int64

## 認知度ラベルと突き合わせ、167件の推薦対象を確定する（03/04と同じ手順）

In [3]:
awareness = pd.read_csv("../data/processed/awareness_scores.csv", index_col=0)
name_to_numidx = pd.Series(num.index, index=names)

recommendable_awareness = awareness[awareness["recommendable"]]
rows = []
for job_idx, row in recommendable_awareness.iterrows():
    hit = name_to_numidx.get(row["職業名"])
    if hit is None:
        continue
    if isinstance(hit, pd.Series):
        hit = hit.iloc[0]
    rows.append({"job_id": int(hit), "awareness_score": row["awareness_score"], "awareness_label": row["awareness_label"]})

job_pool = pd.DataFrame(rows).set_index("job_id")
job_pool = job_pool[riasec_source.loc[job_pool.index] != "unavailable"]
print("最終的な推薦対象:", len(job_pool))


最終的な推薦対象: 167


## RIASECの標準化・PCA（05と同じ、167件基準）

In [4]:
riasec_scaler = StandardScaler().fit(riasec.loc[riasec_observed, riasec_cols])

riasec_167 = riasec_full.loc[job_pool.index, riasec_cols]
riasec_167_z = pd.DataFrame(
    riasec_scaler.transform(riasec_167.values), columns=riasec_cols, index=riasec_167.index
)

pca_full = PCA(n_components=6, random_state=0).fit(riasec_167_z.values)
pc_scores = pca_full.transform(riasec_167_z.values)
pc_std = pc_scores.std(axis=0)

print("寄与率:", pca_full.explained_variance_ratio_.round(3))
print("PC標準偏差:", pc_std.round(3))


寄与率: [0.368 0.237 0.183 0.111 0.054 0.046]
PC標準偏差: [1.454 1.167 1.026 0.797 0.557 0.516]


/Users/oobasouma/yumetane/api/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## jobs.csv を組み立てる

In [5]:
desc_name_indexed = desc.set_index(desc[desc.columns[1]])

jobs = pd.DataFrame({
    "job_name": names.loc[job_pool.index],
    "riasec_source": riasec_source.loc[job_pool.index],
    "awareness_score": job_pool["awareness_score"],
    "awareness_label": job_pool["awareness_label"],
})

for c in riasec_cols:
    jobs[f"riasec_{num_labels[c]}"] = riasec_167.loc[jobs.index, c].values
    jobs[f"riasec_{num_labels[c]}_z"] = riasec_167_z.loc[jobs.index, c].values

pc_df = pd.DataFrame(pc_scores[:, :4], columns=["PC1", "PC2", "PC3", "PC4"], index=riasec_167_z.index)
jobs = jobs.join(pc_df)

descriptions = []
for name in jobs["job_name"]:
    if name in desc_name_indexed.index:
        row = desc_name_indexed.loc[name]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        descriptions.append(row.get("IPD_03_01_000", ""))
    else:
        descriptions.append("")
jobs["description"] = descriptions

jobs = jobs.reset_index().rename(columns={"index": "job_id"})
jobs.to_csv("../data/processed/jobs.csv", index=False, encoding="utf-8-sig")
print("保存件数:", len(jobs))
jobs.head()


保存件数: 167


,job_id,job_name,riasec_source,awareness_score,awareness_label,riasec_現実的,riasec_現実的_z,riasec_研究的,riasec_研究的_z,riasec_芸術的,...,riasec_社会的_z,riasec_企業的,riasec_企業的_z,riasec_慣習的,riasec_慣習的_z,PC1,PC2,PC3,PC4,description
0,2,洋菓子製造、パティシエ,observed,2,知っている,3.345,0.022717,3.000,-0.339886,2.945,...,-0.486454,2.836,-0.712261,3.455,0.864462,-0.770855,0.245568,0.403848,0.717076,洋菓子店や菓子工場で洋菓子を製造する。
1,11,ハム・ソーセージ・ベーコン製造,observed,0,知らない,3.304,-0.085748,2.739,-0.931222,2.543,...,-1.321065,2.652,-1.222722,3.326,0.440488,-2.032450,0.299823,-0.127627,0.239784,原料肉を分割・整形した後、塩づけ、くん煙などの加工をして、ハム・ソーセージ・ベーコンを製造する。
2,12,ワイン製造,observed,1,名前は聞いたことがある,3.708,0.983028,3.292,0.321686,2.958,...,-0.203500,3.292,0.552794,3.375,0.601532,0.434715,0.708825,0.822922,0.086022,ブドウからワインを醸造する作業に従事する。
3,13,ビール製造,observed,1,名前は聞いたことがある,3.594,0.681443,3.469,0.722707,3.031,...,-0.329710,3.219,0.350274,3.219,0.088820,0.629905,0.852734,0.291800,-0.124637,ビール醸造所（ブルワリー）において、ビールの製造に従事する。
4,15,野菜つけ物製造,observed,0,知らない,3.382,0.120600,2.818,-0.752235,2.545,...,-1.115465,2.691,-1.114527,3.236,0.144693,-1.738322,0.411827,-0.180489,0.131667,野菜を材料にしたつけ物を製造するため、材料の選別、洗浄、カット、漬け込み、塩抜き、計量、殺菌、検査、包装などの作...


## `riasec_transform.json` を組み立てる

APIが質問の採点〜推薦をnotebookに依存せず行うための変換パラメータ一式。
`notebooks/07_question_randomization.ipynb`の`QUESTIONS`・採点ロジックと
同じ内容をJSONに落とす。


In [6]:
QUESTIONS = [
    {"id": "Q1", "axis": "PC1", "option_a": "テストのように答えが一つに決まっている問題の方が面白い", "option_b": "感想文のように人それぞれ答えが違う問題の方が面白い", "positive": "b"},
    {"id": "Q2", "axis": "PC1", "option_a": "教わったとおりにやる方が好き", "option_b": "自分なりのやり方を試す方が好き", "positive": "b"},
    {"id": "Q3", "axis": "PC1", "option_a": "レシピどおりに正確に作る方が好き", "option_b": "レシピを見ながらも自分なりにアレンジする方が好き", "positive": "b"},
    {"id": "Q4", "axis": "PC2", "option_a": "こわれたものを自分で直してみたい", "option_b": "友達のけんかの仲直りを手伝いたい", "positive": "a"},
    {"id": "Q5", "axis": "PC2", "option_a": "新しい道具や機械のしくみを調べる方が得意な気がする", "option_b": "友達や家族の悩みを聞いてあげる方が得意な気がする", "positive": "a"},
    {"id": "Q6", "axis": "PC2", "option_a": "作り方の動画を見て何かを作る方が楽しそう", "option_b": "人の話を聞いてアドバイスする方が楽しそう", "positive": "a"},
    {"id": "Q7", "axis": "PC3", "option_a": "最初に完成までの手順を決めてから進める方が自分に近い", "option_b": "進めながら決めていく方が自分に近い", "positive": "a"},
    {"id": "Q8", "axis": "PC3", "option_a": "ゲームはルールどおりに進める方が好き", "option_b": "自分たちでルールを変えて遊ぶ方が好き", "positive": "a"},
    {"id": "Q9", "axis": "PC4", "option_a": "表現の美しさ・かっこよさにこだわる授業の方が好き", "option_b": "しくみや理由を突き止める授業の方が好き", "positive": "a"},
    {"id": "Q10", "axis": "PC4", "option_a": "作品を作って見せる方にする", "option_b": "調べて分かったことをまとめる方にする", "positive": "a"},
]
AXIS_N = {"PC1": 3, "PC2": 3, "PC3": 2, "PC4": 2}
SCALE_SD = 1.5

transform = {
    "riasec_cols": [num_labels[c] for c in riasec_cols],
    "riasec_scaler_mean": riasec_scaler.mean_.tolist(),
    "riasec_scaler_scale": riasec_scaler.scale_.tolist(),
    "pca_components": pca_full.components_.tolist(),  # 6x6, 行=PC、列=riasec_cols順
    "pca_mean": pca_full.mean_.tolist(),
    "pc_std": pc_std.tolist(),
    "scale_sd": SCALE_SD,
    "axis_n": AXIS_N,
    "axis_to_pc_index": {"PC1": 0, "PC2": 1, "PC3": 2, "PC4": 3},
    "questions": QUESTIONS,
}

with open("../data/processed/riasec_transform.json", "w", encoding="utf-8") as f:
    json.dump(transform, f, ensure_ascii=False, indent=2)

print("保存しました。キー:", list(transform.keys()))


保存しました。キー: ['riasec_cols', 'riasec_scaler_mean', 'riasec_scaler_scale', 'pca_components', 'pca_mean', 'pc_std', 'scale_sd', 'axis_n', 'axis_to_pc_index', 'questions']


## 検証：JSONの変換パラメータだけで、07の符号チェックと同じ結果が出るか

`api/`側の実装がこのJSONだけを見て動くことを前提にするので、ここで独立に読み込んで
07と同じ結果になるか確認する。


In [7]:
with open("../data/processed/riasec_transform.json", encoding="utf-8") as f:
    t = json.load(f)

def score_from_json(answers: dict, t: dict) -> np.ndarray:
    raw = {axis: 0 for axis in t["axis_n"]}
    for q in t["questions"]:
        chosen = answers[q["id"]]
        ab = "a" if chosen.strip() == q["option_a"].strip() else "b"
        sign = 1 if ab == q["positive"] else -1
        raw[q["axis"]] += sign

    pc_vec = np.zeros(len(t["pc_std"]))
    for axis, score in raw.items():
        idx = t["axis_to_pc_index"][axis]
        frac = score / t["axis_n"][axis]
        pc_vec[idx] = frac * t["scale_sd"] * t["pc_std"][idx]

    components = np.array(t["pca_components"])
    mean = np.array(t["pca_mean"])
    riasec_z = pc_vec @ components + mean
    return riasec_z


# session1（06で使った回答）で再現できるか確認
session1_answers = {
    "Q1": "テストのように答えが一つに決まっている問題の方が面白い",
    "Q2": "教わったとおりにやる方が好き",
    "Q3": "レシピどおりに正確に作る方が好き",
    "Q4": "こわれたものを自分で直してみたい",
    "Q5": "新しい道具や機械のしくみを調べる方が得意な気がする",
    "Q6": "作り方の動画を見て何かを作る方が楽しそう",
    "Q7": "最初に完成までの手順を決めてから進める方が自分に近い",
    "Q8": "ゲームはルールどおりに進める方が好き",
    "Q9": "しくみや理由を突き止める授業の方が好き",
    "Q10": "作品を作って見せる方にする",
}

riasec_z_from_json = score_from_json(session1_answers, t)
print("JSONだけから復元したRIASEC(z):", riasec_z_from_json.round(2))

expected = np.array([1.89, -0.27, -0.77, -1.43, -1.56, 1.26])
assert np.allclose(riasec_z_from_json, expected, atol=0.02), "以前の計算結果と一致しない"
print("06で計算した結果と一致：JSONだけで採点ロジックを再現できている")


JSONだけから復元したRIASEC(z): [ 1.89 -0.27 -0.77 -1.43 -1.56  1.26]
06で計算した結果と一致：JSONだけで採点ロジックを再現できている
